# DS4 Game Theory (GaT) -- Stage 4 Step 1

**Notebook:** `56_ds4_game_theory.ipynb`  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_gat_blueprint.md`

---

## Paper grounding (OneSec paper SS2.6, quoted in full in the blueprint)

> "The model treats opening versus dismissal as a repeated strategic choice involving the present self and the future self."
>
> "Phenotypes: Users may be described as commitment-oriented, reward-oriented, chronic overriders, adaptive switchers, or friction-avoidant users."
>
> "...payoff and utility are also not directly observed, so the model should be used as an interpretive framework for repeated self-regulation choices rather than as a literal payoff model."

## Goal

Classify participants into the paper's own 4 operationalizable named phenotypes using real, redundancy-checked DS4 quantities. No formal payoff matrix or Nash equilibrium -- matches the paper's own explicit caution.

## Output

```
outputs/model_outputs/ds4_game_theory/
  participant_gat_phenotypes.csv
  gat_chi_ou_correlation.csv
  gat_phenotype_crosstab.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | No formal payoff matrix / Nash equilibrium |
| FLAG 2 | Drop `p_open_after_dismiss` -- near-tautological with opened-rate |
| FLAG 3 | `substitution_after_dismissal_rate` computed fresh (new quantity, not covered by NeT) |
| FLAG 4 | Rule-based classification, not re-clustering -- Step 2 already did formal clustering |
| FLAG 5 | "Friction-avoidant" phenotype not operationalized -- no DS4 quantity distinguishes it from commitment-oriented |


## DS4 structural cautions

- `substitution_after_dismissal_rate` uses attempt rows (excludes `app_close_event`), same convention as NeT.
- No experimental condition -- all cross-checks are correlational, not causal.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency

OUT_DIR = Path("outputs/model_outputs/ds4_game_theory")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
beh = pd.read_csv("outputs/cleaned/ds4_behavioral_clean.csv", parse_dates=["timestamp_utc"], low_memory=False)
feat = pd.read_csv("outputs/features/ds4/ds4_behavior_features.csv")

attempts = beh[beh["intervention_event_class"] != "app_close_event"].copy()
attempts = attempts.sort_values(["participant_id_canonical", "timestamp_utc"])
attempts["app_next"] = attempts.groupby("participant_id_canonical")["app_name"].shift(-1)

dismissed_rows = attempts[attempts["event_outcome_simple"] == "dismissed"].dropna(subset=["app_next"]).copy()
dismissed_rows["substituted"] = dismissed_rows["app_name"] != dismissed_rows["app_next"]

sub_rate = dismissed_rows.groupby("participant_id_canonical")["substituted"].mean().rename(
    "substitution_after_dismissal_rate")
n_dismissals = dismissed_rows.groupby("participant_id_canonical").size().rename(
    "n_dismissal_events_with_next")

print(f"Pooled substitution-after-dismissal rate: {dismissed_rows['substituted'].mean():.4f}")

gat = feat[["participant_id_canonical", "rate_opened_intervention_events",
    "p_open_after_dismiss", "max_consecutive_opens", "open_rate_change_calendar"]].copy()
gat = gat.merge(sub_rate.reset_index(), on="participant_id_canonical", how="left")
gat = gat.merge(n_dismissals.reset_index(), on="participant_id_canonical", how="left")

usable = gat[
    (gat["n_dismissal_events_with_next"] >= 5) &
    gat["rate_opened_intervention_events"].notna() &
    gat["max_consecutive_opens"].notna()
].reset_index(drop=True)
print(f"Usable N: {len(usable)}")


In [ ]:
r_paod = usable["p_open_after_dismiss"].corr(usable["rate_opened_intervention_events"])
r_mco = usable["max_consecutive_opens"].corr(usable["rate_opened_intervention_events"])
r_sub = usable["substitution_after_dismissal_rate"].corr(usable["rate_opened_intervention_events"])
print(f"corr(p_open_after_dismiss, rate_opened) = {r_paod:.4f}  <- dropped, near-tautological")
print(f"corr(max_consecutive_opens, rate_opened) = {r_mco:.4f}  <- kept, moderate")
print(f"corr(substitution_after_dismissal_rate, rate_opened) = {r_sub:.4f}  <- kept, near-independent")


In [ ]:
open_median = usable["rate_opened_intervention_events"].median()
override_tertile = usable["max_consecutive_opens"].quantile(2 / 3)
sub_tertile = usable["substitution_after_dismissal_rate"].quantile(2 / 3)


def classify(row):
    if row["rate_opened_intervention_events"] >= open_median:
        if row["max_consecutive_opens"] >= override_tertile:
            return "chronic_overrider"
        return "reward_oriented"
    else:
        if row["substitution_after_dismissal_rate"] >= sub_tertile:
            return "adaptive_switcher"
        return "commitment_oriented"


usable["gat_phenotype"] = usable.apply(classify, axis=1)
print(f"open_median={open_median:.4f}, override_tertile={override_tertile:.2f}, "
      f"sub_tertile={sub_tertile:.4f}")
print(usable["gat_phenotype"].value_counts())


In [ ]:
stm = pd.read_csv("outputs/model_outputs/ds4_statistical_mechanics/participant_ds4_stm_parameters.csv")
gat_stm = usable.merge(stm[["participant_id_canonical", "chi_OU", "fit_valid"]],
    on="participant_id_canonical", how="left")
valid_stm = gat_stm[gat_stm["fit_valid"] == True]

print(valid_stm.groupby("gat_phenotype")["chi_OU"].describe())

groups = [g["chi_OU"].dropna().values for _, g in valid_stm.groupby("gat_phenotype")]
groups = [g for g in groups if len(g) > 1]
f_stat, p_val = stats.f_oneway(*groups)
print(f"\nANOVA chi_OU by gat_phenotype: F={f_stat:.3f}, p={p_val:.4f}")


In [ ]:
phase1 = pd.read_csv("outputs/model_outputs/ds4_phenotyping/ds4_cluster_assignments.csv")
patha = pd.read_csv("outputs/model_outputs/ds4_phenotyping/path_a_feature_matrix.csv")[
    ["participant_id_canonical", "path_a_cluster"]]

gat_phen = usable.merge(phase1, on="participant_id_canonical", how="left")
gat_phen = gat_phen.merge(patha, on="participant_id_canonical", how="left")

ct_phase1 = pd.crosstab(gat_phen["gat_phenotype"], gat_phen["cluster"])
print("GaT phenotype x Phase 1 cluster:")
print(ct_phase1)

chi2, p, dof, exp = chi2_contingency(ct_phase1)
print(f"\nchi2={chi2:.3f}, dof={dof}, p={p:.4f}")


In [ ]:
adapt_check = usable.dropna(subset=["open_rate_change_calendar"]).copy()
adapt_check["abs_change"] = adapt_check["open_rate_change_calendar"].abs()
print(adapt_check.groupby("gat_phenotype")["abs_change"].describe())

groups2 = [g["abs_change"].dropna().values for _, g in adapt_check.groupby("gat_phenotype")]
groups2 = [g for g in groups2 if len(g) > 1]
f2, p2 = stats.f_oneway(*groups2)
print(f"\nANOVA |open_rate_change_calendar| by gat_phenotype: F={f2:.3f}, p={p2:.4f}")


In [ ]:
usable.to_csv(OUT_DIR / "participant_gat_phenotypes.csv", index=False)

chi_summary = valid_stm.groupby("gat_phenotype")["chi_OU"].agg(
    ["mean", "median", "std", "count"]).reset_index()
chi_summary.to_csv(OUT_DIR / "gat_chi_ou_correlation.csv", index=False)

ct_phase1.to_csv(OUT_DIR / "gat_phenotype_crosstab.csv")

print("Wrote participant_gat_phenotypes.csv, gat_chi_ou_correlation.csv, gat_phenotype_crosstab.csv")


In [ ]:
summary_lines = [
    "# DS4 GaT -- Model Run Summary", "",
    "**Notebook:** `56_ds4_game_theory.ipynb`  ",
    "**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_gat_blueprint.md`  ",
    "**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"), "",
    f"Usable N={len(usable)}", "",
    "## Phenotype distribution", "", "```",
    usable["gat_phenotype"].value_counts().to_string(), "```", "",
    "## Redundancy check", "",
    f"corr(p_open_after_dismiss, rate_opened)={r_paod:.4f} (dropped), "
    f"corr(max_consecutive_opens, rate_opened)={r_mco:.4f} (kept), "
    f"corr(substitution_after_dismissal_rate, rate_opened)={r_sub:.4f} (kept)", "",
    "## Cross-checks", "",
    f"ANOVA chi_OU by gat_phenotype: F={f_stat:.3f}, p={p_val:.4f}",
    f"Chi-square gat_phenotype x Phase1 cluster: chi2={chi2:.3f}, p={p:.4f}",
    f"ANOVA |open_rate_change_calendar| by gat_phenotype: F={f2:.3f}, p={p2:.4f}",
]
with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ["commitment_oriented", "reward_oriented", "chronic_overrider", "adaptive_switcher"]
counts = usable["gat_phenotype"].value_counts().reindex(order)
ax.bar(order, counts.values, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax.set_ylabel("n participants")
ax.set_title(f"DS4 GaT phenotype distribution (N={len(usable)})")
plt.xticks(rotation=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_gat_phenotype_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
valid_stm.boxplot(column="chi_OU", by="gat_phenotype", ax=ax)
ax.set_title(f"chi_OU by GaT phenotype (ANOVA p={p_val:.4f})")
plt.suptitle("")
ax.set_xlabel("")
plt.xticks(rotation=20)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_gat_chiou_by_phenotype.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ct_phase1.values, cmap="Oranges")
ax.set_xticks(range(ct_phase1.shape[1]))
ax.set_xticklabels(ct_phase1.columns)
ax.set_yticks(range(ct_phase1.shape[0]))
ax.set_yticklabels(ct_phase1.index)
for i in range(ct_phase1.shape[0]):
    for j in range(ct_phase1.shape[1]):
        ax.text(j, i, ct_phase1.values[i, j], ha="center", va="center")
ax.set_xlabel("Phase 1 cluster")
ax.set_title(f"GaT phenotype x Phase 1 cluster (chi2 p={p:.4f})")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_gat_vs_phase1_crosstab.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- These are descriptive, rule-based phenotypes matching the paper's own named categories -- not a re-clustering (Step 2 already did formal clustering) and not a formal game-theoretic equilibrium (the paper explicitly cautions against a literal payoff model here).
- Cross-checks are correlational only -- no causal claim about which phenotype "performs better."
- The paper's 5th named phenotype ("friction-avoidant") is not operationalized -- no DS4 quantity distinguishes it from commitment-oriented.


## Forward connections

| Connection | Status |
|---|---|
| DS4 ABM (57) | Independent of this notebook -- uses StoP output directly |
| DS4 Step 2 (phenotyping) | GaT phenotypes could become an additional overlay dimension in a future extended phenotyping pass |
